# Baseline Predictions: R-Score Bands

Map each baseline prediction to its evaluation applicant by `id_candidat`, then compare predicted grant rates for remote and center applicants across five R-score bands. These are model selection rates, not observed scholarship outcomes.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

candidate_relative_path = Path("equialgo-participants/data/candidats_evaluation.csv")
repository_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / candidate_relative_path).is_file()),
    None,
)
if repository_root is None:
    raise FileNotFoundError(f"Could not find {candidate_relative_path} from {Path.cwd()}")

analysis_dir = repository_root / "Analysis" / "fairness_and_bias_analysis" / "baseline_model_predictions"
predictions_path = analysis_dir / "baseline_predictions.csv"
candidates = pd.read_csv(repository_root / candidate_relative_path)
predictions = pd.read_csv(predictions_path).rename(
    columns={"decision_octroi": "baseline_prediction"}
)

if candidates["id_candidat"].duplicated().any() or predictions["id_candidat"].duplicated().any():
    raise ValueError("Candidate and prediction IDs must be unique for one-to-one mapping.")
mapped = candidates.merge(
    predictions, on="id_candidat", how="outer", validate="one_to_one", indicator=True
)
if not mapped["_merge"].eq("both").all():
    raise ValueError("Prediction IDs do not match the evaluation candidate IDs.")
mapped = mapped.drop(columns="_merge")
print(f"Mapped {len(mapped)} baseline predictions to evaluation applicants.")

remote_regions = {"Bas-Saint-Laurent", "Cote-Nord", "Gaspesie-Iles-de-la-Madeleine"}
mapped["region_group"] = mapped["region_administrative"].apply(
    lambda region: "remote" if region in remote_regions else "center"
)
band_labels = ["1 lowest", "2", "3", "4", "5 highest"]
mapped["r_band"] = pd.qcut(mapped["cote_r_equivalent"], q=5, labels=band_labels)

table = (
    mapped.pivot_table(
        index="r_band", columns="region_group", values="baseline_prediction",
        aggfunc="mean", observed=True,
    )
    .reindex(columns=["center", "remote"])
    .mul(100)
)
print("Baseline predicted grant rate (%) by R-score band:")
print(table.round(1).to_string())

ax = table.plot(kind="bar", figsize=(9, 5), width=0.78)
ax.set_title("Baseline Predicted Grant Rate by R-Score Band")
ax.set_xlabel("R-score band (lowest to highest)")
ax.set_ylabel("Predicted grants (%)")
ax.set_ylim(0, 100)
ax.legend(title="Region group")
ax.tick_params(axis="x", rotation=0)
ax.figure.tight_layout()
chart_path = analysis_dir / "baseline_r_band_analysis.png"
ax.figure.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved chart: {chart_path}")